# Data Processing for Large Language Models (LLM)
### สรุปเนื้อหาและตัวอย่างโค้ดประกอบ Chapter 2: Data Processing

สรุปเนื้อหาจากสไลด์ **Chapter 2: Data Processing** และ Exp Code ประกอบ  เพื่ออธิบายการทำงานองแต่ละขั้นตอนใน **LLM Data Pipeline**

**หัวข้อทั้งหมด**

1. Data Processing คืออะไร และทำไมต้องเตรียมข้อมูล
2. LLM Data Pipeline (8 ขั้นตอนของการแปลงข้อมูล)
3. Types of Data (Structured-ข้อมูลที่มีโครสร้าง/ Semi-Structured-กึ่งโครงสร้าง / Unstructured-ไม่มีโครงสร้าง)
4. Data Collection-แหล่งข้อมูลที่ถูกนำมาทดลอง
5. Data Quality and Cleaning-คุณภาพของข้อมูล
6. Text Normalization-การแก้ไขโครงสร้างของข้อมูลให้มีมารตฐาน
7. Duplicate Removal-ลบข้อมูลที่ซ้ำซ้อน
8. Tokenization-การแปลงข้อความย่อย
9. Why Tokens Matter-ทำไมจำนวน Token สำคัญ
10. Text Splitting-การแบ่งข้อความ
11. Data Chunking-แบ่งข้อมูลออกเป็นส่วนๆ
12. Chunk Size and Overlap-ขนาดของ Chunk และ Overlab ต่อ Retrieval
13. Metadata-บริบทและที่มา
14. Preparing Data for Embedding-การ preparing ข้อมูล
15. Embedding-การแปลงข้อความเป็นเวกเตอร์
16. Vector Database และ Retrieval-การจัดเก็บและค้นคืนข้อมูล
17. End-to-End Data Processing Workflow-ภาพรวมกระบวนการทั้งหมด
18. Common Problems-และแนวทางแก้ไข
19. Best Practices-แนวทางปฏิบัติที่ดีในการเตรียมข้อมูล
20. Summary-สรุป

> **Key Message:** Better Data → Better LLM Response


## 0. เตรียมเครื่องมือ (Setup)

ไลบรารี:
- `tiktoken` เพื่อทำ Tokenization ซึ่งเป็นวิธีการพื้นฐานกับการทำงานของ GPT 
- `scikit-learn` เพื่อทำ Embedding และคำนวณ cosine similarity
- `pandas` / `numpy` เพื่อจัดการตัวอย่างข้อมูลในรูปแบบตาราง


In [4]:
# เรียกใช้ไลบรารี
import sys
!{sys.executable} -m pip install --quiet tiktoken scikit-learn pandas numpy
print("เรียกใช้เรียบร้อยแล้ว")

เรียกใช้เรียบร้อยแล้ว



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import re
import json
import html
import hashlib
import unicodedata
import difflib
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80) 
print("เรียกใช้ไลบรารีร้อยแล้ว")

เรียกใช้ไลบรารีร้อยแล้ว


## 1. Data Processing คืออะไร

**Data Processing** คือการแปลงข้อมูลดิบ (raw data) จากแหล่งข้อมูลใดๆ ให้เป็นข้อมูลที่ **มีโครงสร้าง เหมาะสมสำหรับการใช้งานกับ LLM**

**เนื่องจาก**
- ข้อมูลดิบมีความซ้ำซ้อน ไม่สมบูรณ์ หรือมีรูปแบบที่ไม่เหมาะสม
- การเตรียมข้อมูลช่วยให้โมเดลเข้าใจบริบทได้ดีขึ้น
- ช่วยเพิ่มความแม่นยำ ลดการสร้างข้อมูลผิดพลาด (hallucination)

**ความสัมพันธ์ระหว่าง data processing และคุณภาพคำตอบ**

> key message: **better data → better LLM response**


## 2. LLM Data Pipeline (ลำดับขั้นตอนการแปลงข้อมูลให้พร้อมใช้งาน)

Pipeline มี 8 ขั้นตอน:

| ขั้นตอน | ชื่อ | คำอธิบาย |
|---|---|---|
| 1 | **Collection** | รวบรวมข้อมูลจากแหล่งต่าง ๆ เข้ามาไว้ในที่เดียว |
| 2 | **Cleaning** | ลบข้อมูลที่ไม่จำเป็น เช่น HTML, โฆษณา, Header/Footer, ข้อความซ้ำ |
| 3 | **Normalization** | ปรับรูปแบบข้อความให้เป็นมาตรฐาน เช่น ช่องว่าง, Unicode |
| 4 | **Chunking** | แบ่งข้อความออกเป็นชิ้น (Chunks) ที่มีขนาดเหมาะสม |
| 5 | **Metadata** | เพิ่มข้อมูลประกอบ เช่น ชื่อเอกสาร ผู้เขียน วันที่ แหล่งที่มา |
| 6 | **Embedding** | แปลงข้อความเป็นเวกเตอร์ตัวเลข |
| 7 | **Vector Database** | จัดเก็บเวกเตอร์เพื่อการค้นคืนข้อมูล (Retrieval) |
| 8 | **LLM / Retrieval** | นำผลลัพธ์ไปใช้ตอบคำถามอย่างมีคุณภาพ |

**แนวคิดสำคัญ:** ข้อมูลที่ถูกเตรียมอย่างดี จะทำให้ LLM เข้าใจและตอบคำถามได้แม่นยำ น่าเชื่อถือได้มากขึ้น

**ผลลัพธ์:** ข้อมูลที่ผ่านกระบวนการทั้งหมดจะพร้อมสำหรับการค้นหาและใช้งานร่วมกับ LLM เพื่อให้ได้คำตอบที่มีคุณภาพ (Vector Database → Retrieval → LLM ตอบคำถาม)

Exp: Pipeline ทั้ง 8 ขั้นตอนแบบง่าย ๆ ก่อนจะลงรายละเอียดของแต่ละขั้นตอนในหัวข้อถัดไป


In [6]:
pipeline_steps = [
    ("1. Collection", "รวบรวมข้อมูลจากแหล่งต่าง ๆ เข้ามาไว้ในที่เดียว"),
    ("2. Cleaning", "ลบข้อมูลที่ไม่จำเป็น เช่น HTML, โฆษณา, Header/Footer"),
    ("3. Normalization", "ปรับรูปแบบข้อความให้เป็นมาตรฐานเดียวกัน"),
    ("4. Chunking", "แบ่งข้อความเป็นชิ้นที่เหมาะสม (Chunks)"),
    ("5. Metadata", "เพิ่มข้อมูลประกอบเพื่อช่วยในการค้นหา"),
    ("6. Embedding", "แปลงข้อความเป็นเวกเตอร์ตัวเลข"),
    ("7. Vector Database", "จัดเก็บเวกเตอร์เพื่อการค้นคืนข้อมูล"),
    ("8. LLM (ตอบคำถาม)", "สร้างคำตอบที่มีคุณภาพจากข้อมูลที่ค้นคืนมาได้"),
]

df_pipeline = pd.DataFrame(pipeline_steps, columns=["ขั้นตอน", "คำอธิบาย"])
df_pipeline

,ขั้นตอน,คำอธิบาย
0,1. Collection,รวบรวมข้อมูลจากแหล่งต่าง ๆ เข้ามาไว้ในที่เดียว
1,2. Cleaning,"ลบข้อมูลที่ไม่จำเป็น เช่น HTML, โฆษณา, Header/Footer"
2,3. Normalization,ปรับรูปแบบข้อความให้เป็นมาตรฐานเดียวกัน
3,4. Chunking,แบ่งข้อความเป็นชิ้นที่เหมาะสม (Chunks)
4,5. Metadata,เพิ่มข้อมูลประกอบเพื่อช่วยในการค้นหา
5,6. Embedding,แปลงข้อความเป็นเวกเตอร์ตัวเลข
6,7. Vector Database,จัดเก็บเวกเตอร์เพื่อการค้นคืนข้อมูล
7,8. LLM (ตอบคำถาม),สร้างคำตอบที่มีคุณภาพจากข้อมูลที่ค้นคืนมาได้


## 3. Types of Data (ประเภทของข้อมูล)

LLM มีรูปแบบของข้อมูลอยู่ 3 แบบ:

1. **Structured Data** — ข้อมูลที่มีโครงสร้างชัดเจน เก็บในรูปแบบตาราง แถว คอลัมน์ (เช่น SQL/Database, CSV, Excel)
2. **Semi-Structured Data** — ข้อมูลกึ่งโครงสร้าง มีโครงสร้างบางส่วนแต่ไม่เป็นตารางแน่นอน มีแท็กหรือคีย์ช่วยตีความหมาย (เช่น JSON, XML, HTML)
3. **Unstructured Data** — ข้อมูลไม่มีโครงสร้าง ไม่มีรูปแบบตายตัว ต้องแปลงหรือสกัดข้อมูลก่อนใช้งาน (เช่น PDF, DOCX, TXT, Web Page)

Exp: การเข้าใจประเภทของข้อมูลช่วยให้เราเลือกวิธีการประมวลผลที่เหมาะสม และใช้ข้อมูลร่วมกับ LLM ได้อย่างมีประสิทธิภาพ


In [ ]:
# Exp 1. Structured Data - ข้อมูลตาราง
structured_data = pd.DataFrame({
    "ID": ["A001", "A002", "A003"],
    "Name": ["Laptop", "Mouse", "Keyboard"],
    "Amount": [25000, 500, 1200],
    "Date": ["2026-01-01", "2026-01-02", "2026-01-03"],
})
print("1) Structured Data (SQL / CSV / Excel)")
structured_data

1) Structured Data (SQL / CSV / Excel)


,ID,Name,Amount,Date
0,A001,Laptop,25000,2026-01-01
1,A002,Mouse,500,2026-01-02
2,A003,Keyboard,1200,2026-01-03


In [ ]:
# Exp 2. Semi-Structured Data - JSON
semi_structured_data = {
    "id": "A001",
    "name": "Laptop",
    "price": 25000,
    "category": "Computer"
}
print("2) Semi-Structured Data (JSON / XML / HTML)")
print(json.dumps(semi_structured_data, indent=2, ensure_ascii=False))

2) Semi-Structured Data (JSON / XML / HTML)
{
  "id": "A001",
  "name": "Laptop",
  "price": 25000,
  "category": "Computer"
}


In [33]:

# Exp 3. Unstructured Data - ข้อความอิสระจากเอกสาร
unstructured_data = """
บริษัท เอไอ๊ เอไอ จำกัด
คาดการณ์ว่าจะก่อตั้งขึ้นใน ไตรมาสที่ 3 ปี 2569
ทุนจดทะเบียน 1,000,000 บาท
บริษัทตั้งเป้าหมายให้อัตรากำไรสุทธิ (Net Profit Margin) ไม่น้อยกว่าร้อยละ 8 ภายในไตรมาสที่ 3 ปี พ.ศ. 2570 """
print("3) Unstructured Data (PDF / DOCX / TXT / Web Page)")
print(unstructured_data)


3) Unstructured Data (PDF / DOCX / TXT / Web Page)

บริษัท เอไอ๊ เอไอ จำกัด
คาดการณ์ว่าจะก่อตั้งขึ้นใน ไตรมาสที่ 3 ปี 2569
ทุนจดทะเบียน 1,000,000 บาท
บริษัทตั้งเป้าหมายให้อัตรากำไรสุทธิ (Net Profit Margin) ไม่น้อยกว่าร้อยละ 8 ภายในไตรมาสที่ 3 ปี พ.ศ. 2570 


## 4. Data Collection (การเก็บรวบรวมข้อมูล)

**แหล่งข้อมูลที่ใช้ในระบบ LLM ที่พบบ่อย** ได้แก่ Internal Documents, Databases, Websites, APIs, Knowledge Bases, Books & Publications, Public Datasets, User Generated Content

**Key Idea:** เก็บข้อมูลให้ครบ → เลือกข้อมูลที่ดี → ใช้ข้อมูลอย่างมีคุณภาพ ช่วยให้ LLM เข้าใจบริบทได้ดีและตอบได้แม่นยำ

Exp: การรวบรวมข้อมูล (Collection) จากหลายแหล่งเข้ามาไว้ในโครงสร้างเดียวกัน ก่อนนำไปประมวลผลต่อ


In [32]:
import pandas as pd

# ตัวอย่างการรวบรวมข้อมูลจากหลายแหล่ง
raw_documents = [
    {
        "source_type": "Internal Document",
        "title": "คู่มือพนักงาน.pdf",
        "raw_text": "   คู่มือพนักงาน   บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษาความปลอดภัยของข้อมูล   "
    },
    {
        "source_type": "Website",
        "title": "เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด",
        "raw_text": "<div><h1>บริการด้าน AI</h1><p>พัฒนา AI, Web Application, Local LLM &amp; AI Search สำหรับองค์กร</p></div>"
    },
    {
        "source_type": "Database",
        "title": "ฐานข้อมูลโครงการ",
        "raw_text": "ข้อมูลโครงการ   AI Search, Knowledge Base   และ Dashboard สำหรับการบริหารจัดการข้อมูลภายในองค์กร"
    },
    {
        "source_type": "Internal Document",
        "title": "คู่มือพนักงาน.pdf",  # เอกสารซ้ำ (Duplicate)
        "raw_text": "   คู่มือพนักงาน   บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษาความปลอดภัยของข้อมูล   "
    },
]

df_raw = pd.DataFrame(raw_documents)

print(f"ข้อมูลทั้งหมด {len(df_raw)} เอกสาร จากหลากหลายแหล่งข้อมูล")
df_raw

ข้อมูลทั้งหมด 4 เอกสาร จากหลากหลายแหล่งข้อมูล


,source_type,title,raw_text
0,Internal Document,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษ...
1,Website,เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด,"<div><h1>บริการด้าน AI</h1><p>พัฒนา AI, Web Application, Local LLM &amp; AI ..."
2,Database,ฐานข้อมูลโครงการ,"ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัด..."
3,Internal Document,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษ...


## 5. Data Quality and Cleaning (คุณภาพข้อมูลและการทำความสะอาดข้อมูล)

**ปัญหาคุณภาพข้อมูลที่พบบ่อย:**
1. Duplicate — ข้อมูลซ้ำซ้อน
2. Noise — ข้อมูลที่ไม่เกี่ยวข้อง เช่น โฆษณา, ข้อความแทรก
3. Broken Text — ข้อความเสียหายจากการแปลงไฟล์ หรือ OCR ไม่สมบูรณ์
4. Missing Content — เนื้อหาบางส่วนหายไป
5. HTML Tags — แท็ก HTML หรือแท็กที่ไม่จำเป็นปนอยู่ในเอกสาร
6. Header / Footer — ส่วนหัวและท้ายเอกสารที่ซ้ำทุกหน้า
7. Irrelevant Information — ข้อมูลที่ไม่เกี่ยวข้องกับเป้าหมายของระบบ
8. Inconsistent Format — รูปแบบข้อมูลไม่สม่ำเสมอ

**ทำไมต้องใส่ใจคุณภาพข้อมูล:** ข้อมูลที่ไม่ดี = คำตอบที่ไม่ดี, เพิ่มความเสี่ยงของ Hallucination, ลดความแม่นยำของ Retrieval, สิ้นเปลืองทรัพยากร

Exp: ฟังก์ชัน `clean_text()` สำหรับลบ HTML tag, ช่องว่างเกิน, และอักขระที่ไม่จำเป็นออกจากข้อความดิบ


In [34]:
import re
import html

def clean_text(text: str) -> str:
    """ทำความสะอาดข้อความจากเอกสาร เว็บไซต์ และฐานข้อมูล
    - แปลง HTML entities
    - ลบ HTML tags
    - ลบช่องว่างและบรรทัดว่างส่วนเกิน
    """
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

# ทำความสะอาดข้อมูลในคอลัมน์ raw_text
df_raw["cleaned_text"] = df_raw["raw_text"].apply(clean_text)

# แสดงข้อความก่อนและหลังทำความสะอาด
df_raw[["source_type", "title", "raw_text", "cleaned_text"]]

,source_type,title,raw_text,cleaned_text
0,Internal Document,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษ...,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\nนโยบายการใช้ระบบ AI และการรักษาความปล...
1,Website,เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด,"<div><h1>บริการด้าน AI</h1><p>พัฒนา AI, Web Application, Local LLM &amp; AI ...","บริการด้าน AI พัฒนา AI, Web Application, Local LLM & AI Search สำหรับองค์กร"
2,Database,ฐานข้อมูลโครงการ,"ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัด...","ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัดการข..."
3,Internal Document,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\n\nนโยบายการใช้ระบบ AI และการรักษ...,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\nนโยบายการใช้ระบบ AI และการรักษาความปล...


## 6. Text Normalization (การปรับข้อความให้เป็นมาตรฐาน)

**Key Idea:** ทำให้ข้อความ "สะอาด" และ "เป็นมาตรฐานเดียวกัน" เพื่อลดความคลาดเคลื่อนและเพิ่มคุณภาพของข้อมูล

**ขั้นตอนในการทำ Text Normalization:**
1. ลบช่องว่างเกิน (Remove Extra Spaces)
2. ลบอักขระพิเศษ (Remove Special Characters) ที่ไม่จำเป็น เช่น `! @ # $ % ^ & *`
3. ปรับมาตรฐานอักขระ Unicode (Unicode Normalization) เช่น การรวมรูปแบบ é ที่มีหลายรูปแบบให้เป็นรูปแบบเดียวกัน (NFC)
4. จัดรูปแบบตัวอักษร (Standard Formatting) เช่น เครื่องหมายคำพูด, ยัติภังค์

Exp: `" AI   is   AMAZING!!! "` → `"AI is amazing!"`, `"café" (หลายรูปแบบ)` → `"café"` (มาตรฐานเดียวกัน)


In [54]:
import re
import unicodedata

def normalize_text(text: str, remove_special_chars: bool = True) -> str:
    """ปรับข้อความให้เป็นมาตรฐาน (Normalization)
    1. Unicode Normalization (NFC) - รวมอักขระที่มีหลายรูปแบบให้เป็นรูปแบบเดียวกัน
    2. ลบช่องว่างส่วนเกิน
    3. ลบอักขระพิเศษที่ไม่จำเป็น (เก็บเฉพาะตัวอักษร ตัวเลข วรรคตอนพื้นฐาน และภาษาไทย)
    """
    # 1. Unicode normalization (เช่นทำให้ é ที่เขียนได้หลายแบบ กลายเป็นแบบเดียวกัน)
    text = unicodedata.normalize("NFC", text)

    # 2. ลบช่องว่างซ้ำ / ช่องว่างต้น-ท้ายบรรทัด
    text = re.sub(r"\s+", " ", text).strip()

    # 3. ลบอักขระพิเศษที่ไม่จำเป็น (คงไว้เฉพาะตัวอักษรไทย/อังกฤษ ตัวเลข และเครื่องหมายวรรคตอนพื้นฐาน)
    if remove_special_chars:
        text = re.sub(r"[^\w\sก-๙.,!?:;\-\/]", "", text, flags=re.UNICODE)
        text = re.sub(r"\s+", " ", text).strip()

    return text

# ตัวอย่างข้อความจากเอกสาร เว็บไซต์ และฐานข้อมูล
examples = [
    "   คู่มือพนักงาน   บริษัท เอไอ๊ เอไอ จำกัด   ",
    "บริการด้าน AI,   Web Application,   Local LLM!!!",
    "ระบบ AI Search &amp; Knowledge Base สำหรับองค์กร®",
    "ข้อมูลโครงการ AI Search, Dashboard™ และ Inventory Management ©2026",
]

for ex in examples:
    print(f"ก่อน : {ex!r}")
    print(f"หลัง : {normalize_text(ex)!r}")
    print("-" * 60)

ก่อน : '   คู่มือพนักงาน   บริษัท เอไอ๊ เอไอ จำกัด   '
หลัง : 'คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด'
------------------------------------------------------------
ก่อน : 'บริการด้าน AI,   Web Application,   Local LLM!!!'
หลัง : 'บริการด้าน AI, Web Application, Local LLM!!!'
------------------------------------------------------------
ก่อน : 'ระบบ AI Search &amp; Knowledge Base สำหรับองค์กร®'
หลัง : 'ระบบ AI Search amp; Knowledge Base สำหรับองค์กร'
------------------------------------------------------------
ก่อน : 'ข้อมูลโครงการ AI Search, Dashboard™ และ Inventory Management ©2026'
หลัง : 'ข้อมูลโครงการ AI Search, Dashboard และ Inventory Management 2026'
------------------------------------------------------------


In [55]:
# นำไปใช้กับข้อมูลที่ผ่านการ cleaning มาแล้ว
df_raw["normalized_text"] = df_raw["cleaned_text"].apply(normalize_text)
df_raw[["title", "cleaned_text", "normalized_text"]]

,title,cleaned_text,normalized_text
0,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\nนโยบายการใช้ระบบ AI และการรักษาความปล...,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด นโยบายการใช้ระบบ AI และการรักษาความปลอ...
1,เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด,"บริการด้าน AI พัฒนา AI, Web Application, Local LLM & AI Search สำหรับองค์กร","บริการด้าน AI พัฒนา AI, Web Application, Local LLM AI Search สำหรับองค์กร"
2,ฐานข้อมูลโครงการ,"ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัดการข...","ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัดการข..."
3,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด\nนโยบายการใช้ระบบ AI และการรักษาความปล...,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด นโยบายการใช้ระบบ AI และการรักษาความปลอ...


## 7. Duplicate Removal (การลบข้อมูลซ้ำ)

**เหตุผลที่ต้องลบข้อมูลซ้ำ:** ลดขนาดฐานข้อมูล, ลด Token Cost, ลด Retrieval ซ้ำ, เพิ่มคุณภาพ Embedding

**แนวคิดในการตรวจจับข้อมูลซ้ำ:**
1. **Exact Duplicate** — ข้อมูลที่เหมือนกันทุกประการ (อักขระ/คำ/เว้นวรรค/รูปแบบ เหมือนกันทั้งหมด)
2. **Near Duplicate** — ข้อมูลที่คล้ายกันมาก แต่มีความแตกต่างเล็กน้อย (เช่น คำพ้อง, รูปแบบต่างกัน, ตัวเลข/วันที่ต่างกันเล็กน้อย)

Exp: การตรวจจับทั้งสองแบบ: Exact duplicate ใช้การเปรียบเทียบ hash ของข้อความ ส่วน Near duplicate ใช้ค่าความคล้าย (similarity ratio)


In [57]:
def get_text_hash(text: str) -> str:
    """สร้าง hash ของข้อความ สำหรับตรวจจับ Exact Duplicate"""
    normalized = normalize_text(text).lower()
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()

def remove_exact_duplicates(documents: list, text_key: str = "normalized_text") -> list:
    """ลบเอกสารที่ซ้ำกัน โดยเก็บไว้เพียง 1 ชิ้น"""
    seen_hashes = set()
    unique_docs = []
    for doc in documents:
        h = get_text_hash(doc[text_key])
        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_docs.append(doc)
    return unique_docs

documents = df_raw.to_dict("records")
print(f"จำนวนเอกสารก่อนลบ: {len(documents)}")

unique_documents = remove_exact_duplicates(documents)
print(f"จำนวนเอกสารหลังลบ: {len(unique_documents)}")

pd.DataFrame(unique_documents)[["title", "normalized_text"]]

จำนวนเอกสารก่อนลบ: 4
จำนวนเอกสารหลังลบ: 3


,title,normalized_text
0,คู่มือพนักงาน.pdf,คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด นโยบายการใช้ระบบ AI และการรักษาความปลอ...
1,เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด,"บริการด้าน AI พัฒนา AI, Web Application, Local LLM AI Search สำหรับองค์กร"
2,ฐานข้อมูลโครงการ,"ข้อมูลโครงการ AI Search, Knowledge Base และ Dashboard สำหรับการบริหารจัดการข..."


In [ ]:
import difflib

def is_near_duplicate(text_a: str, text_b: str, threshold: float = 0.9) -> bool:
    """ตรวจสอบ near duplicate ด้วยค่าความคล้ายของข้อความ
    ใช้ difflib.SequenceMatcher ที่หมาะสำหรับข้อความที่สั้นๆ (เช่นเอกสารสั้น ๆ หรือข้อความจากเว็บไซต์)
    """
    ratio = difflib.SequenceMatcher(None, text_a, text_b).ratio()
    return ratio >= threshold, ratio

doc_a = "บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI, Web Application, Local LLM และระบบ AI Search สำหรับองค์กร"
doc_b = "บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI, Web Application, Local LLM รวมถึงระบบ AI Search สำหรับองค์กร"
doc_c = "คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด นโยบายการใช้ระบบ AI และการรักษาความปลอดภัยของข้อมูล"

for name, (x, y) in {
    "A vs B (มีความใกล้เคียงกัน)": (doc_a, doc_b),
    "A vs C (มีความต่างกัน)": (doc_a, doc_c)
}.items():
    is_dup, score = is_near_duplicate(x, y, threshold=0.5)
    print(f"{name}: similarity = {score:.2f} -> Near Duplicate: {is_dup}")

A vs B (มีความใกล้เคียงกัน): similarity = 0.96 -> Near Duplicate: True
A vs C (มีความต่างกัน): similarity = 0.42 -> Near Duplicate: False


## 8. Tokenization (แปลงข้อความเป็นหน่วยย่อย)

ขั้นตอนก่อนที่ LLM จะประมวลผล Text โดยที่ Text จะถูกแปลงเป็นลำดับของ **Tokens** ซึ่งเป็นหน่วยพื้นฐานที่โมเดลเข้าใจ

**ทำไม LLM ไม่อ่านข้อความโดยตรง**
- LLM ทำงานกับตัวเลข ไม่ใช่ข้อความ
- การแปลงเป็น Tokens ช่วยให้โมเดลประมวลผลได้อย่างมีประสิทธิภาพ
- ช่วยจัดการ word ที่ไม่เคยเห็นมาก่อน (Out-of-Vocabulary) ได้ดีขึ้น
- ช่วยควบคุมขนาดโมเดลและหน่วยความจำ

**Subword Tokenization** เป็นวิธีที่นิยมใช้ใน LLM (เช่น BPE, WordPiece, SentencePiece) การใช้งานอาศัยการแบ่งออกเป็น คำๆ หรือ คำย่อย (Subword) ตามความถี่ของการใช้งาน

% ------------------------------------------------------

Exp: การทำ Tokenization ด้วย `tiktoken` ซึ่งเป็น Tokenizer แบบ BPE ที่ใช้จริงในโมเดลตระกูล GPT


In [63]:
import re
import tiktoken

# ใช้ Encoding แบบเดียวกับโมเดลตระกูล GPT-4 / GPT-3.5
# หมายเหตุ: การเรียกใช้ครั้งแรกต้องดาวน์โหลดตาราง BPE ผ่านอินเทอร์เน็ต
# หากเครื่องไม่มีอินเทอร์เน็ต หรือถูกบล็อกโดยไฟร์วอลล์
# จะสลับไปใช้ Tokenizer อย่างง่ายแทน เพื่อสาธิตแนวคิดเดียวกัน

USE_TIKTOKEN = True

try:
    encoding = tiktoken.get_encoding("cl100k_base")

except Exception as e:
    USE_TIKTOKEN = False

    print(
        f"ไม่สามารถโหลด tiktoken ('cl100k_base') ได้ ({type(e).__name__}) "
        "-> จะใช้ Tokenizer อย่างง่ายแทนเพื่อสาธิตแนวคิดเดียวกัน\n"
    )

    class SimpleTokenizer:
        """Tokenizer อย่างง่ายสำหรับสาธิต
        แบ่งข้อความออกเป็นคำและสัญลักษณ์
        (แนวคิดคล้าย Subword Tokenization แบบง่าย)
        """

        def __init__(self):
            self.vocab = {}
            self.inverse_vocab = {}

        def _split(self, text):
            return re.findall(
                r"[ก-๙]+|[A-Za-z]+|[0-9]+|[^\sก-๙A-Za-z0-9]",
                text
            )

        def encode(self, text):
            tokens = self._split(text)
            ids = []

            for tok in tokens:
                if tok not in self.vocab:
                    idx = len(self.vocab)
                    self.vocab[tok] = idx
                    self.inverse_vocab[idx] = tok
                ids.append(self.vocab[tok])

            return ids

        def decode(self, ids):
            return "".join(self.inverse_vocab.get(i, "") for i in ids)

    encoding = SimpleTokenizer()

# ตัวอย่างข้อความจากข้อมูลของบริษัท
sample_text = (
    "บริษัท เอไอ๊ เอไอ จำกัด พัฒนา AI Search, "
    "Web Application และ Local LLM สำหรับองค์กร"
)

token_ids = encoding.encode(sample_text)
tokens_decoded = [encoding.decode([tid]) for tid in token_ids]

print(f"ใช้ tiktoken (cl100k_base): {USE_TIKTOKEN}")
print(f"ข้อความต้นฉบับ             : {sample_text}")
print(f"จำนวน Tokens             : {len(token_ids)}")
print(f"Token IDs                : {token_ids}")
print(f"Tokens (แย word)         : {tokens_decoded}")

ใช้ tiktoken (cl100k_base): True
ข้อความต้นฉบับ             : บริษัท เอไอ๊ เอไอ จำกัด พัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร
จำนวน Tokens             : 53
Token IDs                : [37242, 23084, 31885, 3098, 102, 24152, 36984, 95582, 23780, 50856, 23780, 8321, 232, 95582, 23780, 50856, 23780, 220, 61516, 76169, 26265, 24152, 38133, 220, 60984, 24152, 3098, 240, 20795, 21437, 15592, 7694, 11, 5000, 7473, 220, 72409, 32882, 73367, 8949, 445, 11237, 220, 36748, 76169, 40272, 23084, 84646, 23780, 31534, 41427, 56870, 90832]
Tokens (แย word)         : ['บ', 'ร', 'ิ', '�', '�', 'ั', 'ท', ' เ', 'อ', 'ไ', 'อ', '�', '�', ' เ', 'อ', 'ไ', 'อ', ' ', 'จ', 'ำ', 'ก', 'ั', 'ด', ' ', 'พ', 'ั', '�', '�', 'น', 'า', ' AI', ' Search', ',', ' Web', ' Application', ' ', 'แ', 'ล', 'ะ', ' Local', ' L', 'LM', ' ', 'ส', 'ำ', 'ห', 'ร', 'ับ', 'อ', 'ง', 'ค', '์', 'กร']


In [64]:
import pandas as pd
# เปรียบเทียบจำนวน Token ของข้อความภาษาไทยและภาษาอังกฤษ

comparisons = [
    (
        "ภาษาไทย",
        "บริษัท เอไอ๊ เอไอ จำกัด พัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร"
    ),
    (
        "English",
        "AI-AI Co., Ltd. develops AI Search, Web Applications, and Local LLM solutions for enterprises."
    ),
]

rows = []
for lang, text in comparisons:
    n_tokens = len(encoding.encode(text))
    rows.append({
        "ภาษา": lang, "ข้อความ": text, "จำนวน Token": n_tokens, "จำนวนตัวอักษร": len(text)
    })

pd.DataFrame(rows)

,ภาษา,ข้อความ,จำนวน Token,จำนวนตัวอักษร
0,ภาษาไทย,"บริษัท เอไอ๊ เอไอ จำกัด พัฒนา AI Search, Web Application และ Local LLM สำหรั...",53,83
1,English,"AI-AI Co., Ltd. develops AI Search, Web Applications, and Local LLM solution...",22,94


## 9. Why Tokens Matter (ทำไมจำนวน Token จึงสำคัญ)

จำนวน Token ส่งผลโดยตรงต่อ:
1. **Context Window** — จำนวน Token กำหนดขนาดของบริบทที่โมเดลรับเข้าได้ ถ้าเกินขีดจำกัด โมเดลอาจตัดข้อมูลเก่าออก
2. **Processing Time** — Token มาก = โมเดลต้องประมวลผลมากขึ้น = เวลานานขึ้น
3. **Cost** — ค่าใช้จ่ายของ LLM มักคิดตามจำนวน Token (Input + Output)
4. **Memory Usage** — Token มาก = ใช้หน่วยความจำ (RAM/VRAM) มากขึ้น

**Key Idea:** ยิ่งใช้ Token น้อยเท่าที่จำเป็น ยิ่งได้ผลลัพธ์เร็วขึ้น ประหยัดขึ้น และใช้ทรัพยากรน้อยลง แต่ยังคงคุณภาพของคำตอบ

Exp: การประมาณ **ต้นทุน (Cost)** และ **เวลาในการประมวลผล** โดยประมาณจากจำนวน Token (ตัวเลขราคาเป็นเพียงตัวอย่างสมมติเพื่อการสาธิตเท่านั้น)


In [65]:
def estimate_token_cost(text: str, price_per_1k_tokens: float = 0.002) -> dict:
    """ประมาณจำนวน Token และค่าใช้จ่ายโดยสมมติราคาต่อ 1,000 Token
    (ตัวเลขราคาเป็นตัวอย่างสมมติเพื่อการสาธิตเท่านั้น ไม่ใช่ราคาจริง)
    """
    n_tokens = len(encoding.encode(text))
    cost = (n_tokens / 1000) * price_per_1k_tokens
    return {"จำนวน Token": n_tokens, "ต้นทุนโดยประมาณ (USD)": round(cost, 6)}

long_text = " ".join([sample_text] * 50)  # จำลองข้อความยาวขึ้น 50 เท่า
short_text = sample_text

for name, t in [("ข้อความสั้น", short_text), ("ข้อความยาว (x50)", long_text)]:
    result = estimate_token_cost(t)
    print(f"{name}: {result}")

ข้อความสั้น: {'จำนวน Token': 53, 'ต้นทุนโดยประมาณ (USD)': 0.000106}
ข้อความยาว (x50): {'จำนวน Token': 2699, 'ต้นทุนโดยประมาณ (USD)': 0.005398}


## 10. Text Splitting (การแบ่งข้อความเป็นหน่วยย่อยอย่างเหมาะสม)

การแบ่งเอกสารขนาดใหญ่ให้เป็นหน่วยย่อย ช่วยให้ LLM เข้าใจและค้นหาข้อมูลได้ดีขึ้น ลดความซับซ้อน และเพิ่มประสิทธิภาพ

**ระดับการแบ่งข้อความ (จากใหญ่ไปเล็ก):**
1. **Document** — เอกสารทั้งฉบับ (ขนาดใหญ่ที่สุด) เช่น คู่มือการใช้งานระบบ 100 หน้า
2. **Section** — แบ่งตามหัวข้อหลักหรือส่วนใหญ่ (ขนาดปานกลาง) เช่น บทที่ 1 บทนำ 10 หน้า
3. **Paragraph** — แบ่งตามย่อหน้า (ขนาดเล็ก) เช่น 1 ย่อหน้า 3-5 บรรทัด
4. **Sentence** — แบ่งตามประโยค (ขนาดเล็กที่สุด) เช่น 1 ประโยค 1 หน่วยความคิด

Exp: การแบ่งข้อความในระดับ **Paragraph** และ **Sentence**


In [66]:
import re

def split_into_paragraphs(text: str) -> list:
    """แบ่งข้อความเป็นย่อหน้า โดยใช้บรรทัดว่างเป็นตัวแบ่ง"""
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    return paragraphs


def split_into_sentences(text: str) -> list:
    """แบ่งข้อความเป็นประโยคแบบง่าย
    โดยใช้เครื่องหมาย . ! ? และการเว้นวรรคภาษาไทยเป็นตัวช่วย
    """
    sentences = re.split(r"(?<=[.!?])\s+|(?<=[ก-๙])\s{2,}", text)
    return [s.strip() for s in sentences if s.strip()]


sample_document = """
บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร
ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้าง Knowledge Base เพื่อช่วยตอบคำถามและสนับสนุนการตัดสินใจ

AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหาข้อมูลที่เกี่ยวข้องก่อนส่งให้ Large Language Model ประมวลผล
แนวทางนี้ช่วยเพิ่มความถูกต้องของคำตอบ ลดการสร้างข้อมูลที่ไม่ถูกต้อง และรองรับการใช้งานกับเอกสารขององค์กร
"""

paragraphs = split_into_paragraphs(sample_document)

print(f"จำนวนย่อหน้า: {len(paragraphs)}")
for i, p in enumerate(paragraphs, 1):
    print(f"  [ย่อหน้าที่ {i}] {p[:80]}...")

print()

sentences = split_into_sentences(sample_document)

print(f"จำนวนประโยค: {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"  [ประโยคที่ {i}] {s}")

จำนวนย่อหน้า: 4
  [ย่อหน้าที่ 1] บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM ...
  [ย่อหน้าที่ 2] ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้...
  [ย่อหน้าที่ 3] AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหาข้อมูลที่เกี่...
  [ย่อหน้าที่ 4] แนวทางนี้ช่วยเพิ่มความถูกต้องของคำตอบ ลดการสร้างข้อมูลที่ไม่ถูกต้อง และรองรับการ...

จำนวนประโยค: 2
  [ประโยคที่ 1] บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร
ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้าง Knowledge Base เพื่อช่วยตอบคำถามและสนับสนุนการตัดสินใจ
  [ประโยคที่ 2] AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหาข้อมูลที่เกี่ยวข้องก่อนส่งให้ Large Language Model ประมวลผล
แนวทางนี้ช่วยเพิ่มความถูกต้องของคำตอบ ลดการสร้างข้อมูลที่ไม่ถูกต้อง และรองรับการใช้งานกับเอกสารขององค์กร


## 11. Data Chunking (การแบ่งข้อมูลออกเป็นชิ้น)

**Key Idea:** แบ่งข้อมูลให้ "พอดี" ไม่สั้นเกินไปจนขาดบริบท และไม่ยาวเกินไปจนเกินขีดจำกัดของโมเดล (Token Limit)

**ทำไมต้องทำ Chunking:** ช่วยให้ LLM เข้าใจและจดจำบริบทของข้อมูลได้ดีขึ้น เพิ่มประสิทธิภาพในการค้นหา (Retrieval) รองรับข้อจำกัดของจำนวน Token ที่โมเดลประมวลผลได้ และลดความซ้ำซ้อน

**แนวทางการทำ Chunking:**
1. **Fixed Size** — แบ่งตามจำนวนตัวอักษร/Token ที่กำหนด เช่น 500 Token
2. **Paragraph** — แบ่งตามย่อหน้าหรือโครงสร้างของเอกสาร เหมาะกับเอกสารทั่วไป
3. **Semantic / Meaning** — แบ่งตามความหมาย โดยรักษาบริบทของเนื้อหาไว้ เหมาะกับการค้นหาเชิงความหมาย
4. **Structure** — แบ่งตามโครงสร้าง เช่น หัวข้อ, หัวข้อย่อย, ตาราง, รายการ เหมาะกับเอกสารเชิงธุรกิจ

**Node:** กำหนดขนาด Chunk ให้เหมาะสมกับโมเดลและวัตถุประสงค์การใช้งาน การใช้ Overlap ระหว่าง Chunk (เช่น 10-20%) เพื่อรักษาบริบท และทดสอบ/ปรับแต่งขนาด Chunk เพื่อผลลัพธ์ที่ดีที่สุด (โดยมากจดอยู่ที่ 300-800 Token)


In [ ]:
def chunk_by_fixed_size(text: str, chunk_size: int = 80) -> list:
    """แบ่งข้อความตามจำนวนตัวอักษรคงที่ (Fixed-Size Chunking)
    โดยไม่มีการซ้อนทับ (Overlap)
    """
    return [
        text[i:i + chunk_size].strip()
        for i in range(0, len(text), chunk_size)
        if text[i:i + chunk_size].strip()
    ]

def chunk_by_paragraph(text: str) -> list:
    """แบ่งข้อความตามย่อหน้า (Paragraph-Based Chunking)"""
    return split_into_paragraphs(text)

# ทดลองแบ่งข้อความด้วย 2 วิธี
fixed_chunks = chunk_by_fixed_size(sample_document, chunk_size=80)
paragraph_chunks = chunk_by_paragraph(sample_document)

print(f"=== Fixed-Size Chunking (80 ตัวอักษรต่อ Chunk) : {len(fixed_chunks)} Chunks ===")
for i, chunk in enumerate(fixed_chunks, 1):
    print(f"\nChunk {i}")
    print(chunk)

print("\n" + "=" * 80)

print(f"=== Paragraph-Based Chunking : {len(paragraph_chunks)} Chunks ===")
for i, chunk in enumerate(paragraph_chunks, 1):
    print(f"\nChunk {i}")
    print(chunk)

=== Fixed-Size Chunking (80 ตัวอักษรต่อ Chunk) : 6 Chunks ===

Chunk 1
บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM

Chunk 2
สำหรับองค์กร
ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็

Chunk 3
ว พร้อมทั้งสร้าง Knowledge Base เพื่อช่วยตอบคำถามและสนับสนุนการตัดสินใจ

AI Sear

Chunk 4
ch ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหาข้อมูลที่เกี่ยวข้องก

Chunk 5
่อนส่งให้ Large Language Model ประมวลผล
แนวทางนี้ช่วยเพิ่มความถูกต้องของคำตอบ ลด

Chunk 6
การสร้างข้อมูลที่ไม่ถูกต้อง และรองรับการใช้งานกับเอกสารขององค์กร

=== Paragraph-Based Chunking : 4 Chunks ===

Chunk 1
บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร

Chunk 2
ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้าง Knowledge Base เพื่อช่วยตอบคำถามและสนับสนุนการตัดสินใจ

Chunk 3
AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหาข้อมูลที่เกี่ยวข้องก่อนส่งให้ Large Language M

## 12. Chunk Size and Overlap (ขนาดของ Chunk และการทำ Overlap)

ขนาดของ Chunk และการทำ Overlap ส่งผลต่อคุณภาพการค้นคืน (Retrieval):

| ขนาด Chunk | บริบทที่ครอบคลุม | ความละเอียดของข้อมูล | ประสิทธิภาพการค้นคืน | ค่าใช้จ่าย (Token) |
|---|---|---|---|---|
| 256 Tokens | น้อย | สูงมาก | ปานกลาง | ต่ำ |
| 512 Tokens | พอเหมาะ | สูง | สูง | ปานกลาง |
| 1024 Tokens | มาก | ปานกลาง | ปานกลาง | สูง |

**Overlap คือการซ้อนทับข้อมูลบางส่วนระหว่าง Chunk เพื่อให้บริบทต่อเนื่อง ลดปัญหาข้อมูลสำคัญอยู่ขอบ (Boundary Problem)

**Boundary Problem** คือปัญหาที่ ข้อมูลสำคัญถูกตัดอยู่ตรงรอยต่อ (Boundary) ระหว่างสอง Chunk ทำให้แต่ละ Chunk มีข้อมูลไม่ครบ ส่งผลให้ระบบค้นหา (Retrieval) หรือ LLM ตอบคำถามได้ไม่ถูกต้อง

**Sliding Window** คือเทคนิคการสร้าง Chunk โดยกำหนดขนาด (Size) และการซ้อนทับ (Overlap) แล้วเลื่อนหน้าต่างไปทีละก้าว (Step Size = Size - Overlap)

Exp: ฟังก์ชัน Sliding Window Chunking ที่รองรับการกำหนด `chunk_size` และ `overlap` ได้ พร้อมเปรียบเทียบผลลัพธ์เมื่อ Overlap ต่างกัน


In [68]:
def sliding_window_chunk(text: str, chunk_size: int = 100, overlap: int = 20) -> list:
    """แบ่งข้อความแบบ Sliding Window

    Parameters
    ----------
    chunk_size : จำนวนตัวอักษรต่อ Chunk
    overlap    : จำนวนตัวอักษรที่ซ้อนทับกันระหว่าง Chunk
    step        = chunk_size - overlap
    """
    if overlap >= chunk_size:
        raise ValueError("overlap ต้องน้อยกว่า chunk_size")

    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

# ใช้ข้อความตัวอย่างจากบริษัทเดียวกับตัวอย่างก่อนหน้า
long_sample = sample_document.replace("\n", " ")
# เปรียบเทียบการแบ่ง Chunk เมื่อใช้ Overlap ต่างกัน
settings = [
    ("ไม่มี Overlap (0%)", 0), ("Overlap 25%", 25), ("Overlap 50%", 50),
]
for title, overlap in settings:
    chunks = sliding_window_chunk(
        long_sample,
        chunk_size=100,
        overlap=overlap
    )

    print("=" * 80); print(f"{title}"); print(f"Chunk Size = 100 ตัวอักษร | Overlap = {overlap}")
    print(f"จำนวน Chunks = {len(chunks)}"); print("-" * 80)

    for i, chunk in enumerate(chunks, 1):
        print(f"Chunk {i}")
        print(chunk)
        print()

ไม่มี Overlap (0%)
Chunk Size = 100 ตัวอักษร | Overlap = 0
จำนวน Chunks = 5
--------------------------------------------------------------------------------
Chunk 1
บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร ระบบสา

Chunk 2
มารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้าง Knowledge Base เพื่อช่ว

Chunk 3
ยตอบคำถามและสนับสนุนการตัดสินใจ  AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื่อค้นหา

Chunk 4
ข้อมูลที่เกี่ยวข้องก่อนส่งให้ Large Language Model ประมวลผล แนวทางนี้ช่วยเพิ่มความถูกต้องของคำตอบ ลด

Chunk 5
การสร้างข้อมูลที่ไม่ถูกต้อง และรองรับการใช้งานกับเอกสารขององค์กร

Overlap 25%
Chunk Size = 100 ตัวอักษร | Overlap = 25
จำนวน Chunks = 6
--------------------------------------------------------------------------------
Chunk 1
บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร ระบบสา

Chunk 2
l LLM สำหรับองค์กร ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และ

## 13. Metadata (ข้อมูลเกี่ยวกับข้อมูล)

**Metadata** คือ "ข้อมูลที่อธิบายคุณลักษณะของข้อมูล" (Data about Data) ทำหน้าที่บันทึกบริบทและรายละเอียดของเอกสาร เช่น ชื่อเอกสาร แหล่งที่มา ผู้เขียน วันที่สร้าง หมวดหมู่ ภาษา เวอร์ชัน และสิทธิ์การเข้าถึง แม้ Metadata จะไม่ใช่เนื้อหาหลักของเอกสาร แต่มีบทบาทสำคัญในการจัดเก็บ จัดการ ค้นหา และค้นคืนข้อมู

**ตัวอย่าง Metadata ที่นิยมใช้:** Document Name, Author, Source, Date, Category, Page Number

**บทบาทของ Metadata ในการค้นคืนข้อมูล:** เพิ่มความแม่นยำในการค้นหา, ใช้ตัวกรอง (Filter) เพื่อลดขอบเขตผลลัพธ์, เพิ่มความน่าเชื่อถือและตรวจสอบแหล่งที่มาได้, จัดลำดับความเกี่ยวข้องของผลลัพธ์, สนับสนุนการวิเคราะห์และสรุปผล

Exp: การสร้าง Metadata ให้กับแต่ละ Chunk และการใช้ Metadata เป็นตัวกรอง (Filter) ในการค้นหา


In [69]:
from datetime import datetime

def create_metadata(document_name: str, source: str, chunk_index: int,
                     total_chunks: int, category: str = "General",
                     author: str = "Unknown") -> dict:
    """สร้าง Metadata สำหรับ Chunk หนึ่งชิ้น"""
    return {
        "document_name": document_name,
        "author": author,
        "source": source,
        "date": datetime.now().strftime("%Y-%m-%d"),
        "category": category,
        "chunk_index": chunk_index,
        "total_chunks": total_chunks,
    }

# สร้าง Chunk พร้อม Metadata จากเอกสารตัวอย่าง
demo_chunks = chunk_by_paragraph(sample_document)

chunks_with_metadata = []
for idx, chunk_text in enumerate(demo_chunks, start=1):
    meta = create_metadata(
        document_name="Employee_Handbook.pdf",
        source="Internal Document",
        chunk_index=idx,
        total_chunks=len(demo_chunks),
        category="AI / Knowledge Base",
        author="AI-AI Co., Ltd.",
    )

    chunks_with_metadata.append({
        "text": chunk_text,
        "metadata": meta
    })

for item in chunks_with_metadata:
    print("Chunk:", item["text"][:60], "...")
    print("Metadata:", item["metadata"])
    print("-" * 60)

Chunk: บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Applic ...
Metadata: {'document_name': 'Employee_Handbook.pdf', 'author': 'AI-AI Co., Ltd.', 'source': 'Internal Document', 'date': '2026-07-15', 'category': 'AI / Knowledge Base', 'chunk_index': 1, 'total_chunks': 4}
------------------------------------------------------------
Chunk: ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่าง ...
Metadata: {'document_name': 'Employee_Handbook.pdf', 'author': 'AI-AI Co., Ltd.', 'source': 'Internal Document', 'date': '2026-07-15', 'category': 'AI / Knowledge Base', 'chunk_index': 2, 'total_chunks': 4}
------------------------------------------------------------
Chunk: AI Search ใช้เทคนิค Retrieval-Augmented Generation (RAG) เพื ...
Metadata: {'document_name': 'Employee_Handbook.pdf', 'author': 'AI-AI Co., Ltd.', 'source': 'Internal Document', 'date': '2026-07-15', 'category': 'AI / Knowledge Base', 'chunk_index': 3, 'total_chunks': 4}
-----------------------------------------

In [70]:
# ตัวอย่างการใช้ Metadata เป็นตัวกรอง (Filter) ในการค้นหา

knowledge_base = [
    {"text": "คู่มือพนักงานเกี่ยวกับนโยบายการใช้ระบบ AI และการรักษาความปลอดภัยของข้อมูล...",
     "metadata": {"category": "HR", "date": "2026-01-15", "source": "Internal Document"}},
    {"text": "คู่มือการพัฒนาระบบ AI Search และ Knowledge Base สำหรับองค์กร...",
     "metadata": {"category": "AI", "date": "2026-02-20", "source": "Internal Document"}},
    {"text": "แนวทางการพัฒนา Web Application และ Local LLM สำหรับองค์กร...",
     "metadata": {"category": "AI", "date": "2026-03-10", "source": "Website"}},
]

def filter_by_metadata(kb: list, category: str = None, min_date: str = None, source: str = None) -> list:
    """กรองเอกสารด้วยเงื่อนไข Metadata"""
    results = kb
    if category:
        results = [d for d in results if d["metadata"]["category"] == category]
    if min_date:
        results = [d for d in results if d["metadata"]["date"] >= min_date]
    if source:
        results = [d for d in results if d["metadata"]["source"] == source]
    return results

query_condition = {
    "category": "AI",
    "min_date": "2026-01-01",
    "source": "Internal Document"
}

filtered = filter_by_metadata(knowledge_base, **query_condition)

print(f"เงื่อนไขค้นหา: {query_condition}")
print(f"พบเอกสารที่ตรงเงื่อนไข {len(filtered)} รายการ:")
for d in filtered:
    print(" -", d["text"][:50], "...", d["metadata"])

เงื่อนไขค้นหา: {'category': 'AI', 'min_date': '2026-01-01', 'source': 'Internal Document'}
พบเอกสารที่ตรงเงื่อนไข 1 รายการ:
 - คู่มือการพัฒนาระบบ AI Search และ Knowledge Base สำ ... {'category': 'AI', 'date': '2026-02-20', 'source': 'Internal Document'}


## 14. Preparing Data for Embedding (การเตรียมข้อมูลก่อนสร้าง Embedding)

**Key:** "ข้อมูลเข้า = คุณภาพของผลลัพธ์" ยิ่งข้อมูลสะอาด ชัดเจน และมีบริบทดี ผลลัพธ์จากการค้นคืน (Retrieval) ยิ่งแม่นยำ

**5 ขั้นตอนก่อนการสร้าง Embedding:**
1. **Clean Text** — ทำความสะอาดข้อความ (ลบอักขระพิเศษ, HTML, สคริปต์)
2. **Normalized Text** — ทำให้ข้อความเป็นมาตรฐาน (แปลงตัวพิมพ์เล็ก, จัดรูปแบบวันที่/ตัวเลข)
3. **Good Chunks** — แบ่งข้อความเป็น Chunks ที่ดี (ขนาดเหมาะสม เนื้อหาสมบูรณ์ในตัวเอง)
4. **Metadata** — ใส่ข้อมูลประกอบให้ครบถ้วน (ช่วยกรอง ค้นหา และให้เหตุผล)
5. **Quality Validation** — ตรวจสอบคุณภาพข้อมูล (encoding, ความซ้ำซ้อน, ความยาว)

Exp: ขั้นตอนที่ทำมาก่อนหน้านี้ (Clean → Normalize → Chunk → Metadata) เข้าเป็นฟังก์ชันไปป์ไลน์เดียว พร้อมขั้นตอนตรวจสอบคุณภาพ (Quality Validation) ก่อนนำไปสร้าง Embedding


In [71]:
def validate_chunk_quality(chunk_text: str, min_length: int = 10, max_length: int = 2000) -> dict:
    """ตรวจสอบคุณภาพข้อมูลของ Chunk ก่อนนำไปสร้าง Embedding"""
    issues = []
    if len(chunk_text) < min_length:
        issues.append("Chunk สั้นเกินไป (อาจขาดบริบท)")
    if len(chunk_text) > max_length:
        issues.append("Chunk ยาวเกินไป (อาจเกินขีดจำกัดของโมเดล)")
    try:
        chunk_text.encode("utf-8").decode("utf-8")
    except UnicodeError:
        issues.append("พบปัญหาการเข้ารหัส (Encoding Error)")
    return {"is_valid": len(issues) == 0, "issues": issues, "length": len(chunk_text)}

def prepare_for_embedding(raw_text: str, document_name: str, source: str,
                          category: str = "General", chunk_size: int = 100,
                          overlap: int = 20) -> list:
    """ไปป์ไลน์เตรียมข้อมูลก่อนสร้าง Embedding ครบทุกขั้นตอน:
    Clean -> Normalize -> Chunk -> Metadata -> Quality Validation
    """
    # 1-2. Clean + Normalize
    cleaned = clean_text(raw_text)
    normalized = normalize_text(cleaned, remove_special_chars=False)

    # 3. Chunking (Sliding Window)
    chunks = sliding_window_chunk(normalized, chunk_size=chunk_size, overlap=overlap)

    # 4-5. Metadata + Quality Validation
    prepared_chunks = []
    for idx, chunk_text in enumerate(chunks, start=1):
        metadata = create_metadata(document_name, source, idx, len(chunks), category)
        quality = validate_chunk_quality(chunk_text)
        prepared_chunks.append({
            "text": chunk_text,
            "metadata": metadata,
            "quality": quality,
        })
    return prepared_chunks

ready_chunks = prepare_for_embedding(
    raw_text=raw_documents[1]["raw_text"],
    document_name=raw_documents[1]["title"],
    source=raw_documents[1]["source_type"],
    category="AI / Knowledge Base",
    chunk_size=60,
    overlap=15,
)

print(f"เตรียมข้อมูลสำเร็จ ได้ {len(ready_chunks)} chunks ที่พร้อมสำหรับ Embedding\n")
pd.DataFrame([
    {
        "chunk_index": c["metadata"]["chunk_index"],
        "text": c["text"],
        "is_valid": c["quality"]["is_valid"]
    }
    for c in ready_chunks
])

เตรียมข้อมูลสำเร็จ ได้ 2 chunks ที่พร้อมสำหรับ Embedding



,chunk_index,text,is_valid
0,1,"บริการด้าน AI พัฒนา AI, Web Application, Local LLM & AI Sear",True
1,2,l LLM & AI Search สำหรับองค์กร,True


## 15. Embedding (การแปลงข้อความเป็นเวกเตอร์)

**Embedding** คือกระบวนการแปลงข้อความให้เป็นเวกเตอร์ตัวเลข (Numeric Vector) ที่สื่อถึงความหมายของข้อความนั้น ๆ ทำให้คอมพิวเตอร์สามารถคำนวณ "ความใกล้เคียงทางความหมาย" ระหว่างข้อความสองชิ้นได้

ในการใช้งานจริง มักใช้โมเดล Embedding เฉพาะทาง เช่น `text-embedding-3-small` ของ OpenAI หรือโมเดลตระกูล Sentence-Transformers

Exp: การรันให้รวดเร็วโดยไม่ต้องพึ่งพาโมเดลภายนอก สามารถใช้เทคนิค **TF-IDF (Term Frequency - Inverse Document Frequency)** จาก `scikit-learn` เป็น "Embedding" ได้ ซึ่งเป็นตัวอย่างเดียวกันใน **การแปลงข้อความเป็นเวกเตอร์ที่เอาความคล้ายกันมาคำนวณ**


In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# รวบรวมข้อความจาก Chunks ที่ผ่านการเตรียมข้อมูลแล้ว
corpus_texts = [chunk["text"] for chunk in ready_chunks]

# สร้าง Embedding ด้วย TF-IDF 
vectorizer = TfidfVectorizer()
embedding_matrix = vectorizer.fit_transform(corpus_texts)

print(f"จำนวน Chunks              : {embedding_matrix.shape[0]}")
print(f"มิติของ Embedding Vector : {embedding_matrix.shape[1]} (จำนวนคำศัพท์ที่ไม่ซ้ำกัน)")
print()

if embedding_matrix.shape[0] > 0:
    print("ตัวอย่าง Embedding ของ Chunk แรก (แสดงเฉพาะค่าที่ไม่เป็นศูนย์บางส่วน):")

    first_vector = embedding_matrix[0].toarray()[0]
    feature_names = vectorizer.get_feature_names_out()

    nonzero_idx = np.nonzero(first_vector)[0][:10]

    for idx in nonzero_idx:
        print(f"   คำ: {feature_names[idx]:<20} ค่า TF-IDF: {first_vector[idx]:.4f}")

จำนวน Chunks              : 2
มิติของ Embedding Vector : 14 (จำนวนคำศัพท์ที่ไม่ซ้ำกัน)

ตัวอย่าง Embedding ของ Chunk แรก (แสดงเฉพาะค่าที่ไม่เป็นศูนย์บางส่วน):
   คำ: ai                   ค่า TF-IDF: 0.5906
   คำ: application          ค่า TF-IDF: 0.2767
   คำ: llm                  ค่า TF-IDF: 0.1969
   คำ: local                ค่า TF-IDF: 0.2767
   คำ: sear                 ค่า TF-IDF: 0.2767
   คำ: web                  ค่า TF-IDF: 0.2767
   คำ: การด                 ค่า TF-IDF: 0.2767
   คำ: ฒนา                  ค่า TF-IDF: 0.2767
   คำ: บร                   ค่า TF-IDF: 0.2767
   คำ: าน                   ค่า TF-IDF: 0.2767


## 16. Vector Database และ Retrieval (การจัดเก็บและค้นคืนข้อมูล)

**Vector Database** ใช้จัดเก็บเวกเตอร์ (Embedding) พร้อม Metadata เพื่อรองรับการค้นคืนข้อมูล (Retrieval) ที่รวดเร็วและแม่นยำ

หลักการค้นคืน (Retrieval) คือ:
1. แปลงคำถาม (Query) ของผู้ใช้ให้เป็นเวกเตอร์ด้วยวิธี Embedding แบบเดียวกับที่ใช้สร้าง Vector Database
2. คำนวณค่าความคล้าย (เช่น **Cosine Similarity**) ระหว่างเวกเตอร์ของคำถามกับเวกเตอร์ของ Chunk ทั้งหมด
3. เลือก Chunk ที่มีค่าความคล้ายสูงที่สุด (Top-K) ส่งกลับไปให้ LLM ใช้ตอบคำถาม

Exp: สร้าง Vector Database ใน memory พร้อมฟังก์ชัน `search` ด้วย Cosine Similarity


In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class SimpleVectorDatabase:
    """Vector Database อย่างง่าย เก็บ Embedding และ Metadata ไว้ในหน่วยความจำ (In-Memory)
    ใช้สาธิตหลักการทำงานเดียวกับ Vector Database จริง เช่น FAISS, Pinecone และ Qdrant
    """
    def __init__(self, vectorizer: TfidfVectorizer):
        self.vectorizer = vectorizer
        self.chunks = []       # เก็บข้อความของแต่ละ Chunk
        self.metadatas = []    # เก็บ Metadata ของแต่ละ Chunk
        self.vectors = None    # เก็บ Embedding Matrix

    def add(self, chunks: list):
        texts = [c["text"] for c in chunks]
        self.chunks.extend(texts)
        self.metadatas.extend([c["metadata"] for c in chunks])
        self.vectors = self.vectorizer.transform(self.chunks)

    def search(self, query: str, top_k: int = 3) -> list:
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.vectors)[0]
        top_indices = similarities.argsort()[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                "text": self.chunks[idx],
                "metadata": self.metadatas[idx],
                "similarity_score": round(float(similarities[idx]), 4),
            })
        return results


# สร้าง Vector Database และเพิ่มข้อมูล Chunk ที่เตรียมไว้
vector_db = SimpleVectorDatabase(vectorizer)
vector_db.add(ready_chunks)

# ทดลองค้นหา (Retrieval)
query = "บริษัทให้บริการพัฒนาระบบ AI อะไรบ้าง"
results = vector_db.search(query, top_k=2)

print(f"คำถาม (Query): {query}\n")
print("ผลลัพธ์ที่ค้นคืนได้ (Top-K Retrieval):")

for i, r in enumerate(results, 1):
    print(f"\n[{i}] Similarity Score: {r['similarity_score']}")
    print(f"    Text: {r['text']}")
    print(f"    Metadata: {r['metadata']}")

คำถาม (Query): บริษัทให้บริการพัฒนาระบบ AI อะไรบ้าง

ผลลัพธ์ที่ค้นคืนได้ (Top-K Retrieval):

[1] Similarity Score: 0.4586
    Text: บริการด้าน AI พัฒนา AI, Web Application, Local LLM & AI Sear
    Metadata: {'document_name': 'เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด', 'author': 'Unknown', 'source': 'Website', 'date': '2026-07-15', 'category': 'AI / Knowledge Base', 'chunk_index': 1, 'total_chunks': 2}

[2] Similarity Score: 0.1065
    Text: l LLM & AI Search สำหรับองค์กร
    Metadata: {'document_name': 'เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด', 'author': 'Unknown', 'source': 'Website', 'date': '2026-07-15', 'category': 'AI / Knowledge Base', 'chunk_index': 2, 'total_chunks': 2}


## 17. End-to-End Data Processing Workflow (ภาพรวมกระบวนการทั้งหมด)

ภาพรวมของดังที่กล่าวมา เพื่อจำลอง Workflow ทั้งหมดตั้งแต่ **PDF (Raw Data)** จนถึง **Vector Database พร้อมสำหรับการค้นคืนข้อมูล**:

`PDF → Extract Text → Cleaning → Normalization → Chunking → Metadata → Embedding → Vector Database`

Exp: การรันกระบวนการทั้งหมดแบบ End-to-End จากเอกสารดิบ ไปจนถึงการค้นหาคำตอบ (คล้ายกับกรณีศึกษา **Building a Knowledge Base** ในสไลด์)


In [75]:
def end_to_end_pipeline(documents: list, chunk_size: int = 80, overlap: int = 15) -> SimpleVectorDatabase:
    """รันกระบวนการ Data Processing แบบครบวงจร (End-to-End) จากเอกสารดิบหลายชิ้น
    จนถึงการสร้าง Vector Database ที่พร้อมค้นคืนข้อมูล
    """
    all_chunks = []
    for doc in documents:
        chunks = prepare_for_embedding(
            raw_text=doc["raw_text"],
            document_name=doc["title"],
            source=doc["source_type"],
            category=doc.get("category", "General"),
            chunk_size=chunk_size,
            overlap=overlap,
        )
        # เก็บเฉพาะ Chunk ที่ผ่านการตรวจสอบคุณภาพ (Quality Validation)
        valid_chunks = [c for c in chunks if c["quality"]["is_valid"]]
        all_chunks.extend(valid_chunks)

    # ลบ Chunk ที่ซ้ำกัน (Exact Duplicate) ก่อนสร้าง Embedding
    unique_chunks = remove_exact_duplicates(
        [{"normalized_text": c["text"], **c} for c in all_chunks],
        text_key="normalized_text",
    )

    corpus = [c["text"] for c in unique_chunks]
    pipeline_vectorizer = TfidfVectorizer()
    pipeline_vectorizer.fit(corpus)

    db = SimpleVectorDatabase(pipeline_vectorizer)
    db.add(unique_chunks)
    return db


# ใช้เอกสารตัวอย่างชุดเดิมที่รวบรวมไว้ตั้งแต่ขั้นตอน Data Collection
knowledge_base_db = end_to_end_pipeline(raw_documents, chunk_size=70, overlap=15)

print(f"สร้าง Knowledge Base สำเร็จ: {len(knowledge_base_db.chunks)} chunks พร้อมค้นคืนข้อมูล\n")

test_query = "บริษัทให้บริการพัฒนาระบบ AI อะไรบ้าง"
answers = knowledge_base_db.search(test_query, top_k=2)

print(f"คำถาม: {test_query}")
for i, ans in enumerate(answers, 1):
    print(f"\n[อันดับ {i}] คะแนนความคล้าย: {ans['similarity_score']}")
    print(f"  เนื้อหา: {ans['text']}")
    print(f"  ที่มา: {ans['metadata']['document_name']} ({ans['metadata']['source']})")

สร้าง Knowledge Base สำเร็จ: 6 chunks พร้อมค้นคืนข้อมูล

คำถาม: บริษัทให้บริการพัฒนาระบบ AI อะไรบ้าง

[อันดับ 1] คะแนนความคล้าย: 0.3848
  เนื้อหา: บริการด้าน AI พัฒนา AI, Web Application, Local LLM & AI Search สำหรับอ
  ที่มา: เว็บไซต์บริษัท เอไอ๊ เอไอ จำกัด (Website)

[อันดับ 2] คะแนนความคล้าย: 0.2845
  เนื้อหา: คู่มือพนักงาน บริษัท เอไอ๊ เอไอ จำกัด นโยบายการใช้ระบบ AI และการรักษาค
  ที่มา: คู่มือพนักงาน.pdf (Internal Document)


## 18. Common Problems (ปัญหาที่พบบ่อยในกระบวนการเตรียมข้อมูล)

**Key:** คุณภาพของข้อมูลตั้งต้นส่งผลโดยตรงต่อคุณภาพการตอบของ LLM

ปัญหาที่พบบ่อย 5 ประการ:

| # | ปัญหา | ตัวอย่าง | ผลกระทบ |
|---|---|---|---|
| 1 | **Poor Data Quality** | Text ที่มี OCR Error, สัญลักษณ์แปลกปน | ความแม่นยำลดลง |
| 2 | **Wrong Chunk Size** | Chunk 50 Token (สั้นไป) หรือ 3000 Token (ยาวไป) | ขาดบริบท หรือมีสัญญาณรบกวน |
| 3 | **Missing Metadata** | ไม่มี field: source, date, category, page | ค้นหาและกรองข้อมูลไม่ได้ |
| 4 | **Duplicate Documents** | ไฟล์เดียวกันอัปโหลดหลายรอบ หรือจากหลายระบบ | บิดเบือนผลการค้นคืน (Retrieval) |
| 5 | **Incomplete Documents** | ไฟล์ PDF ขาดหน้า, สแกนไม่ครบ, ตัดส่วนท้ายออก | เนื้อหาไม่ต่อเนื่อง คำตอบไม่ครบ |

**แนวทางแก้ไข:** ทำความสะอาดข้อมูล, เลือกขนาด Chunk ให้เหมาะสม, ใส่ Metadata ให้ครบ, ตรวจจับและลบข้อมูลซ้ำ, ตรวจสอบความครบถ้วนของเอกสาร

Exp: ผลกระทบของ **Chunk Size ที่ไม่เหมาะสม** ต่อคุณภาพของบริบทที่ Chunk เลือกหรืออธิบายได้


In [76]:
# สาธิตปัญหา Wrong Chunk Size: เปรียบเทียบ Chunk ที่สั้นเกินไปกับ Chunk ขนาดเหมาะสม
problem_text = (
    "บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร "
    "ระบบสามารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว "
    "พร้อมทั้งสร้าง Knowledge Base เพื่อสนับสนุนการตอบคำถามและการตัดสินใจ"
)

too_small_chunks = chunk_by_fixed_size(problem_text, chunk_size=20)   # เล็กเกินไป -> ขาดบริบท
good_chunks = chunk_by_fixed_size(problem_text, chunk_size=100)       # ขนาดเหมาะสมกว่า

print("ตัวอย่าง Chunk เล็กเกินไป (20 ตัวอักษร) -> บริบทขาดหาย:")
for c in too_small_chunks[:3]:
    print("  -", repr(c))

print("\nตัวอย่าง Chunk ขนาดเหมาะสม (100 ตัวอักษร) -> ยังคงใจความสำคัญไว้ครบถ้วนกว่า:")
for c in good_chunks:
    print("  -", repr(c))

ตัวอย่าง Chunk เล็กเกินไป (20 ตัวอักษร) -> บริบทขาดหาย:
  - 'บริษัท เอไอ๊ เอไอ จำ'
  - 'กัด ให้บริการพัฒนา A'
  - 'I Search, Web Applic'

ตัวอย่าง Chunk ขนาดเหมาะสม (100 ตัวอักษร) -> ยังคงใจความสำคัญไว้ครบถ้วนกว่า:
  - 'บริษัท เอไอ๊ เอไอ จำกัด ให้บริการพัฒนา AI Search, Web Application และ Local LLM สำหรับองค์กร ระบบสาม'
  - 'ารถค้นหาข้อมูลจากเอกสาร เว็บไซต์ และฐานข้อมูลได้อย่างรวดเร็ว พร้อมทั้งสร้าง Knowledge Base เพื่อสนับ'
  - 'สนุนการตอบคำถามและการตัดสินใจ'


In [78]:
# ปัญหา Missing Metadata: เปรียบเทียบ Chunk ที่มี Metadata ครบ vs ไม่ครบ

chunk_missing_metadata = {
    "text": problem_text,
    "metadata": {}
}

chunk_with_metadata = {
    "text": problem_text,
    "metadata": create_metadata(
        document_name="Employee_Handbook.pdf",
        source="Internal Document",
        chunk_index=1,
        total_chunks=1,
        category="AI / Knowledge Base",
        author="AI-AI Co., Ltd."
    )
}

def can_be_filtered(chunk: dict) -> bool:
    required_fields = {"source", "date", "category"}
    return required_fields.issubset(chunk["metadata"].keys())

print(
    "Chunk ที่ไม่มี Metadata -> สามารถใช้ Filter ค้นหาได้หรือไม่:",
    can_be_filtered(chunk_missing_metadata)
)

print(
    "Chunk ที่มี Metadata ครบ -> สามารถใช้ Filter ค้นหาได้หรือไม่:",
    can_be_filtered(chunk_with_metadata)
)

Chunk ที่ไม่มี Metadata -> สามารถใช้ Filter ค้นหาได้หรือไม่: False
Chunk ที่มี Metadata ครบ -> สามารถใช้ Filter ค้นหาได้หรือไม่: True


## 19. Best Practices (แนวทางปฏิบัติที่ดีในการเตรียมข้อมูล)

**Key:** ทำให้ข้อมูล "สะอาด" "ครบถ้วน" และ "เหมาะสม" ก่อนนำไปสร้าง Embedding และจัดเก็บใน Vector Database

1. **ใช้ข้อมูลที่เชื่อถือได้** — เลือกแหล่งข้อมูลที่น่าเชื่อถือ มาจากแหล่งทางการหรือผู้เชี่ยวชาญ และอัปเดตข้อมูลให้เป็นปัจจุบัน
2. **ลบข้อมูลซ้ำ** — ตรวจจับเอกสารหรือเนื้อหาที่ซ้ำ เพื่อป้องกันการกวนผลการค้นหา และลดขนาดข้อมูล/ค่าใช้จ่าย
3. **Normalize Text** — ทำให้ข้อความมีมาตรฐานเดียวกัน จัดการตัวพิมพ์ วรรคตอน ช่องว่าง
4. **เลือก Chunk Size ให้เหมาะสม** — เลือกขนาดให้เหมาะกับลักษณะของข้อมูล สมดุลระหว่างบริบทและความละเอียด ทดลองและปรับตามผลการ Retrieval
5. **เพิ่ม Metadata** — เพิ่มข้อมูลประกอบให้กับแต่ละ Chunk เพื่อช่วยในการกรอง ค้นหา และอ้างอิง
6. **ตรวจสอบคุณภาพข้อมูล** — ตรวจสอบความครบถ้วนและความถูกต้อง ตรวจ Encoding ความซ้ำซ้อน และประเมินผลการค้นคืน

**ผลลัพธ์ที่ได้:** ค้นคืนข้อมูลได้แม่นยำและตรงประเด็น ลดความผิดพลาดและข้อมูลลวง เพิ่มประสิทธิภาพระบบ RAG และให้คำตอบที่น่าเชื่อถือและอ้างอิงได้


## 20. Summary (สรุปประเด็นสำคัญ)

1. **Data Processing เป็นพื้นฐานของทุกระบบ LLM** — ข้อมูลที่ดี ช่วยให้โมเดลเข้าใจเนื้อหาได้อย่างถูกต้อง และตอบคำถามได้แม่นยำ
2. **คุณภาพข้อมูลส่งผลโดยตรงต่อคุณภาพคำตอบ** — ข้อมูลที่สะอาด ถูกต้อง และครบถ้วน ลดความคลาดเคลื่อนและการตอบผิดพลาดของ LLM
3. **Tokenization และ Chunking มีผลต่อ Retrieval** — การแบ่งข้อความและขนาดของ Chunk ที่เหมาะสม ช่วยให้ค้นหาข้อมูลที่เกี่ยวข้องได้ครบถ้วนและตรงประเด็น
4. **Metadata ช่วยให้ค้นหาข้อมูลได้แม่นยำขึ้น** — การใส่ Metadata ที่เหมาะสม (เช่น แหล่งที่มา วันที่ หมวดหมู่) ช่วยกรองและค้นหาข้อมูลได้รวดเร็วและแม่นยำ
5. **ข้อมูลที่ผ่านการเตรียมอย่างเหมาะสมทำให้ Embedding มีประสิทธิภาพ** — ข้อความที่สะอาด เป็นมาตรฐาน และมีบริบทที่ชัดเจน ช่วยให้ Embedding จับความหมายได้ดีขึ้น
6. **Data Processing คือรากฐานของ RAG, AI Search และ AI Agent** — เมื่อกระบวนการเตรียมข้อมูลแข็งแรง ระบบ AI ทั้งการค้นหา การตอบคำถาม และการทำงานอัตโนมัติจะมีประสิทธิภาพสูงสุด

> **Data Processing คือรากฐานของ RAG, AI Search และ AI Agent**


---
### หมายเหตุสำหรับผู้สอน/ผู้เรียน

- Notebook นี้ใช้ **TF-IDF** แทน Embedding Model จริง (เช่น `text-embedding-3-small`, Sentence-Transformers) เพื่อให้รันได้รวดเร็วโดยไม่ต้องดาวน์โหลดโมเดลขนาดใหญ่ แนวคิดเรื่องการแปลงข้อความเป็นเวกเตอร์และคำนวณ Cosine Similarity เพื่อค้นคืนข้อมูลยังคงเหมือนกับการใช้งานจริง
- ฟังก์ชัน `clean_text`, `normalize_text`, `sliding_window_chunk`, `create_metadata`, `prepare_for_embedding` และ `SimpleVectorDatabase` สามารถนำไปดัดแปลงต่อยอดเป็น Lab Assignment สามารถฝึกเติมได้
- การใช้งานจริงกับ RAG ควรเปลี่ยนไปใช้ Vector Database จริง เช่น FAISS, Qdrant, Pinecone หรือ Weaviate และใช้ Embedding Model ที่ผ่านการฝึกมาสำหรับงานด้าน Semantic Search เป็นต้น


In [79]:
#.....